In [1]:

#importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# First Model:
## Step 1: Loading the Data
We import our cleaned_data with all data needed

In [2]:
df  = pd.read_csv('cleaned_dataset.csv', sep=';')

## Step 2: Defining the Goal (Target) and what is usable (Features)
To train an AI, we must tell it what it needs to guess (the **Target**) and what information it is allowed to use (the **Features**). 

Then, we clean the table to remove rows where the delay is missing.

In [3]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We remove rows where the delay is equal to -1 and nan (meaning the data was invalid during cleaning).
df = df[df[target] != -1]
df = df.dropna()

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 3: Translating Text for the Computer (Encoding)
An Artificial Intelligence is a calculating machine: it only understands mathematics. It cannot read words like "Paris" or "Bordeaux". 
We must therefore use a "translator" to convert these station names into numerical codes that the computer can analyze. Encoding of columns where the value isn't usable by the model (strings) into and usable value

In [4]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 4: Creating the AI "Assembly Line"
We are going to set up our prediction model. The chosen algorithm is called a **Random Forest**. 
It combines the output of multiple decision trees to reach a single result

In [5]:
# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 5: Creating the final AI
This is the most important step. We will divide our data into two batches:
- **80% for training:** The AI practices guessing the delays and looks at the real answers to learn from its mistakes.
- **20% for testing:** We hide the answers from the AI and ask it to make its predictions to see if it has understood the logic.

In [6]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 6: Grading (Performance Evaluation)
Now that the AI has taken its exam and made its predictions, we will compare its answers with reality to give it performance grades.

In [7]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")

Mean Absolute Error (MAE) : 2.15 minutes
Mean Squared Error (MSE) : 31.21
R² Score : 0.25


## Conclusion on the first model
For a first model, it's not bad but it's not good either. A R² score of 0.25 mean that the model is slightly better than pur randomness.
This mean that this model can be tuned even more to try to have a better score and so a better prediction model.

# Second Model
We saw previously that our model had a score of 0.25, but we need to take in mind that it only use direct or easily deductible parameters. But with what we have now, maybe we could deduce new parameters to give him. For example, if we know the departure and arrival stations, we can do a mean of scheduled train for this trip during this season. It is not the most accurate parameter but it is still better than we don t give any parameter at all.

## Step 1: Adding new parameters
We can add new parameters like : Number of scheduled trains, Number of cancelled trains, Number of trains delayed at departure, Number of delayed trains at arrival

In [8]:
target = 'Average delay of all trains at arrival'
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season', 'Number of scheduled trains', 'Number of trains delayed > 15min']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

## Step 2 : Encoding

In [9]:
# We list the columns that contain text.
categorical_features = ['Service', 'Departure station', 'Arrival station', 'Season']

# We create an automatic translator (ColumnTransformer). 
# It will transform text columns into numbers (using OneHotEncoder).
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    # "remainder='passthrough'" means we don't touch the other columns that are already numbers
    remainder='passthrough'
)

## Step 3: Creating the AI "Assembly Line"

In [10]:
# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

## Step 4: Creating the final AI

In [11]:
# We randomly split the data: 80% for training and 20% for testing.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# We start the learning process, The AI trains to find logical links between the features (X_train) and the target (y_train).
model.fit(X_train, y_train)
# After training, we ask the model to make predictions based on the test features (X_test).
y_pred = model.predict(X_test)

## Step 5: Grading (Performance Evaluation)

In [12]:
# Mean Absolute Error (MAE): On average, by how many minutes was the model wrong compared to the real delay?
# The lower this number is, the better.
print(f"Mean Absolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f} minutes")
# Mean Squared Error (MSE): It looks like MAE, but it heavily "punishes" the AI when it makes a giant mistake.
print(f"Mean Squared Error (MSE) : {mean_squared_error(y_test, y_pred):.2f}")
# R² Score: It measures how well the model's predictions match the real data.
# A score of 1 means perfect predictions, while a score of 0 means the model is no better than just guessing the average delay and a score below 0 means the model is performing worse than random guessing.
print(f"R² Score : {r2_score(y_test, y_pred):.2f}")

Mean Absolute Error (MAE) : 1.55 minutes
Mean Squared Error (MSE) : 31.52
R² Score : 0.24


## Conclusion
We saw that adding new parameters don t necessarly increase the efficiency of the model so we need to think of others things

# Hyperparameters
One solution to improve the model could be the Hyperparamters. Hyperparameters are parameters the guide how the model must learn. In our case we will use Grid Search. Grid Search is an algorythm that will try every combination of hyperparameters to search the best one.

## Step 1 : Setting up the data
We set up everything the same way that before

In [13]:
target = 'Average delay of all trains at arrival'
#We go back to previous features because we saw that they didn't improve the model
features = ['Service', 'Departure station', 'Arrival station', 'Average journey time', 'Month', 'Year', 'Season']

# We split our table into two parts: 
# X contains only the clues (features)
# y contains only the answers to guess (target)
X = df[features]
y = df[target]

# We create a "Pipeline": it is an automated assembly line.
model = Pipeline(steps=[
    # The data first goes through the preprocessor, which transforms text into numbers.
    ('preprocessor', preprocessor),
    # Then it goes to the regressor, which will learn to make predictions based on the features.
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Step 2 : We setup the grid search to try differents parameters
We need to initiate which parameters the grid search will use and tru to combine

In [14]:
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': [0.1, 1, 'scale', 'auto'],
}
svm = SVC()
grid_search = GridSearchCV(estimator=svm, param_grid=param_grid, cv=5, n_jobs=-1)

## Step 3 : Executing the grid search

Now that everything is setup, we can execute it and see print the results of the grid search 

In [ ]:
# On identifie les colonnes contenant du texte et celles contenant des nombres
categorical_cols = ['Service', 'Departure station', 'Arrival station', 'Season']
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# 1. On crée le préprocesseur pour transformer automatiquement le texte en nombres
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

# 2. On crée le Pipeline qui enchaîne la transformation ET le modèle de régression
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42)) # Remplacement de SVC par RandomForestRegressor
])

# 3. On définit la grille d'hyperparamètres (attention au préfixe model__)
param_grid = {
    'model__n_estimators': [50, 100, 200, 300, 400, 500],
    'model__max_depth': [None, 10, 20, 30, 40, 50],
    'model__min_samples_split': [2, 5, 10, 15, 20],
    'model__min_samples_leaf': [1, 2, 4, 6, 8, 10]
}

# 4. On donne le Pipeline (et non plus SVC) au GridSearchCV
grid_search = GridSearchCV(
    estimator=pipeline, 
    param_grid=param_grid, 
    cv=3, # Vous pouvez augmenter le nombre de plis plus tard
    verbose=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

# 5. Séparation et entraînement
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Le fit va maintenant fonctionner car le Pipeline s'occupe de transformer "National" en nombre !
grid_search.fit(X_train, y_train)

print("Meilleurs paramètres :", grid_search.best_params_)

Fitting 3 folds for each of 1080 candidates, totalling 3240 fits
